In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
team_name="team_lemma"
catalog=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
dbutils.widgets.text("source_path","abfss://raw@schwabdldevsa.dfs.core.windows.net","SOURCE PATH")
dbutils.widgets.text("target_path",f"/Volumes/{catalog}/landing/PWG/")

In [0]:
spark.sql(f"use catalog {catalog}")

In [0]:
batch_id=dbutils.widgets.get("batch_id")
source_path=dbutils.widgets.get("source_path")
target_path=dbutils.widgets.get("target_path")

In [0]:
BATCHDATE_COLS=["batchdate", "batchid"]
CONTROL_DOMAIN_FILES=[
    {
        "file_name": "BatchDate.txt", 
        "landing_name": "batchdate",
        "columns_by_batch":{"1":BATCHDATE_COLS,"2":BATCHDATE_COLS,"3":BATCHDATE_COLS},
        "type":"batchdate_txt"
    }
]

In [0]:
def load_domain(batch_id):
    try:
        print(f"Raw to Landing For Batch {batch_id}")
        for f in CONTROL_DOMAIN_FILES:
            if batch_id in f["columns_by_batch"].keys():
                src=f"{source_path}/Batch{batch_id}/{f['file_name']}"

                if f["type"]=="batchdate_txt":
                   df=spark.read.text(src)
                   df=df.withColumnRenamed("Value","batchdate")
    
                run_id=datetime.now().strftime("%Y%m%d_%H%M%S")
                df=df.withColumn("_landing_ts",current_timestamp())\
                    .withColumn("_batch",lit(batch_id))\
                    .withColumn("_source_file",lit(f['file_name']))\
                    .withColumn("_run_id",lit(run_id))
                df.limit(10).display()
                source_count=df.count()
                print(f"Writing File {f['landing_name']}....")
                df.write.mode("overwrite").parquet(f"{target_path}/Batch{batch_id}/{f['landing_name']}")
                print(f"Write successfully with count {df.count()}")
                target_count=spark.read.parquet(f"{target_path}/Batch{batch_id}/{f['landing_name']}").count()
                    
                # log_pipeline_recon(
                #     spark=spark,
                #     run_id=run_id,
                #     batch_id=batch_id,
                #     domain="CUSTOMER",
                #     table_name=f['landing_name'],
                #     source_layer="raw",
                #     target_layer="landing",
                #     source_count=source_count,
                #     target_count=target_count
                # )
                # log_audit_event(
                #     spark=spark,
                #     run_id=run_id,
                #     batch=batch_id,
                #     layer="landing",
                #     table_name=f['landing_name'],
                #     operation="OVERWRITE",
                #     rows_affected=target_count
                # )
    except Exception as e:
        print(e)
        raise e

In [0]:
load_domain(batch_id)